# GRPO Training Orchestrator: AML Lead Auditor
This notebook manages the end-to-end RL training pipeline for the Global Syndicate Taskforce Lead Auditor LLM.

## Pipeline Overview:
1. **Environment Setup**: Install Unsloth and compatible TRL dependencies.
2. **Server Initialization**: Start the OpenEnv FastAPI server in the background.
3. **Model Loading**: Load Qwen2.5-3B with 4-bit quantization via Unsloth.
4. **RL Training**: Execute Group Relative Policy Optimization (GRPO).
5. **Evaluation**: Plot loss and reward curves.

In [ ]:
# 1. Setup Dependencies
# Fixed installation syntax for Unsloth and compatible TRL version
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install "trl>=0.18.2" transformers accelerate peft datasets requests matplotlib

## 2. Environment Server Initialization
We launch the `OpenEnv` server as a background process to provide the RL rewards.

In [ ]:
import subprocess
import sys
import time
import requests

def start_env_server(url="http://127.0.0.1:8000"):
    try:
        requests.get(f"{url}/health", timeout=2).raise_for_status()
        print("Environment server is already running.")
    except Exception:
        print("Starting OpenEnv FastAPI server...")
        # Using a slightly more robust subprocess call
        subprocess.Popen(
            [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        # Allow time for server to boot
        for i in range(10):
            time.sleep(2)
            try:
                requests.get(f"{url}/health", timeout=2).raise_for_status()
                print(f"Server successfully started at {url}")
                return
            except Exception:
                print(f"Waiting for server... attempt {i+1}/10")
        raise RuntimeError("Environment server failed to start.")

start_env_server()

## 3. Model & Trainer Configuration
We import the core logic from `train_grpo_unsloth.py` to maintain consistency between script and notebook.

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import json
import re
import matplotlib.pyplot as plt
from pathlib import Path

# Import utilities from the existing script
from train_grpo_unsloth import SYSTEM_PROMPT, extract_json_candidate, build_reward_functions, build_dataset

# Hyperparameters
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_STEPS = 120
OUTPUT_DIR = "grpo_trained_auditor_nb"
PLOT_DIR = "docs/nb_plots"
LORA_RANK = 16
ENV_URL = "http://127.0.0.1:8000"

print("Configurations loaded.")

## 4. Training Execution
This cell initializes the model, sets up the GRPO trainer, and begins the training loop.

In [ ]:
# Initialize Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset = build_dataset()

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    bf16=False,
    fp16=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=128,
    max_steps=MAX_STEPS,
    save_steps=MAX_STEPS,
    max_grad_norm=0.1,
    report_to="none",
    use_vllm=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=build_reward_functions(ENV_URL),
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

## 5. Result Visualization
Extract and plot the reward and loss curves from the trainer state.

In [ ]:
def plot_results(output_dir, plot_dir):
    # Try to find the most recent checkpoint
    checkpoints = sorted(Path(output_dir).glob("checkpoint-*/trainer_state.json"))
    if not checkpoints:
        print("No trainer_state.json found.")
        return
    
    state_path = checkpoints[-1]
    state = json.loads(state_path.read_text())
    logs = state.get("log_history", [])
    
    steps, losses = [], []
    reward_steps, rewards = [], []
    
    for row in logs:
        step = row.get("step")
        if step is None: continue
        if row.get("loss") is not None:
            steps.append(step)
            losses.append(float(row["loss"]))
        if row.get("reward") is not None:
            reward_steps.append(step)
            rewards.append(float(row["reward"]))
    
    out = Path(plot_dir)
    out.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(steps, losses, color="blue")
    plt.title("Loss Curve")
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(reward_steps, rewards, color="green")
    plt.title("Reward Curve")
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(out / "training_results.png")
    plt.show()

plot_results(OUTPUT_DIR, PLOT_DIR)